# Measure Early-Stage RIR Model Performance

This notebook is now focused on **early-stage RIR reconstruction**. It can train the configured models, export fresh early-window prediction `.npz` files, and evaluate/visualize the first 80 ms of each RIR.

Expected file contract for the evaluation stage:
- required arrays: `y_true`, `y_pred`
- optional arrays: `sample_id`, `method_name`, `sample_rate`, `target_mode`, `early_ms`, `output_dim`
- default shapes: `(n_samples, early_n_points)`

It reports early-waveform metrics (`MAE`, `RMSE`, `R^2`) and early-acoustic metrics (`early_energy`, energy error, peak timing error, and peak amplitude error). The traditional geometric image-source baseline `traditional_way_baseline` is included for comparison with the learned early-RIR models.

The next notebook cells define the GPU training/export configuration and run the pipeline before the evaluation cells load `./result/*_early80ms_predictions.npz`. If you only want to re-plot existing results, set `RUN_TRAINING = False` and `RUN_EXPORT = False` in the config cell.


## Train And Export Early-RIR Pipeline

Run this notebook from top to bottom. The next two code cells will optionally train the neural models, create the traditional geometry baseline checkpoint, and export fresh early-stage prediction files before the evaluation cells load `./result/*_early80ms_predictions.npz`.

If you only want to re-plot existing results later, set `RUN_TRAINING = False` and `RUN_EXPORT = False` in the config cell below.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
TRAIN_PYTHON_EXECUTABLE = Path(r"D:\Anaconda\envs\ad448\python.exe")
TRAIN_MODELS = ["base_model", "rir_small", "rir_large", "rir_depth", "traditional_way_baseline"]
TRAIN_EPOCHS = 20
TRAIN_BATCH_SIZE = 16
TRAIN_LEARNING_RATE = 1e-3
TRAIN_DEVICE = "cuda"
TARGET_MODE = "early"
EARLY_MS = 80.0
TARGET_SUFFIX = f"early{int(EARLY_MS)}ms" if float(EARLY_MS).is_integer() else f"early{str(EARLY_MS).replace('.', 'p')}ms"
RUN_TRAINING = True
RUN_EXPORT = True
KMP_DUPLICATE_LIB_OK = "TRUE"

if not TRAIN_PYTHON_EXECUTABLE.exists():
    raise FileNotFoundError(f"Training Python executable not found: {TRAIN_PYTHON_EXECUTABLE}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Training Python: {TRAIN_PYTHON_EXECUTABLE}")
print(f"Models: {TRAIN_MODELS}")
print(f"Target: {TARGET_MODE}, early_ms={EARLY_MS}, suffix={TARGET_SUFFIX}")
print(f"Device: {TRAIN_DEVICE}")
print("Progress bars are shown when `tqdm` is installed in the training environment.")


In [ ]:
import time


def _run_pipeline_command(command):
    env = os.environ.copy()
    env["KMP_DUPLICATE_LIB_OK"] = KMP_DUPLICATE_LIB_OK
    printable = " ".join(str(part) for part in command)
    print(f"Running command: {printable}")
    started_at = time.perf_counter()
    subprocess.run(command, check=True, cwd=PROJECT_ROOT, env=env)
    elapsed_seconds = time.perf_counter() - started_at
    print(f"Command finished in {elapsed_seconds / 60.0:.2f} min")


if RUN_TRAINING:
    train_command = [
        str(TRAIN_PYTHON_EXECUTABLE),
        "scripts/train_models.py",
        "--models",
        *TRAIN_MODELS,
        "--epochs",
        str(TRAIN_EPOCHS),
        "--batch-size",
        str(TRAIN_BATCH_SIZE),
        "--learning-rate",
        str(TRAIN_LEARNING_RATE),
        "--device",
        TRAIN_DEVICE,
        "--target-mode",
        TARGET_MODE,
        "--early-ms",
        str(EARLY_MS),
    ]
    _run_pipeline_command(train_command)
else:
    print("Skipping training because RUN_TRAINING=False")

if RUN_EXPORT:
    export_command = [
        str(TRAIN_PYTHON_EXECUTABLE),
        "scripts/export_predictions.py",
        "--models",
        *TRAIN_MODELS,
        "--batch-size",
        str(TRAIN_BATCH_SIZE),
        "--device",
        TRAIN_DEVICE,
        "--target-mode",
        TARGET_MODE,
        "--early-ms",
        str(EARLY_MS),
    ]
    _run_pipeline_command(export_command)
else:
    print("Skipping export because RUN_EXPORT=False")

print("Early-RIR training/export pipeline step completed.")


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

RESULT_FILES = [
    Path(f"./result/base_model_{TARGET_SUFFIX}_predictions.npz"),
    Path(f"./result/rir_small_{TARGET_SUFFIX}_predictions.npz"),
    Path(f"./result/rir_large_{TARGET_SUFFIX}_predictions.npz"),
    Path(f"./result/rir_depth_{TARGET_SUFFIX}_predictions.npz"),
    Path(f"./result/traditional_way_baseline_{TARGET_SUFFIX}_predictions.npz"),
]
SAMPLE_RATE = 22050
EPSILON = 1e-12
PLOT_EXAMPLES = True
PLOT_ALL_MODELS = True
BEST_MODEL_METRIC = "early_rmse"
EXAMPLE_INDEX = 0
EXPORT_CSV = True
OUTPUT_DIR = Path("./result")

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
def _as_path(path_like):
    return path_like if isinstance(path_like, Path) else Path(path_like)


def _ensure_2d(array, name, file_path):
    array = np.asarray(array)
    if array.ndim == 1:
        array = array[None, :]
    if array.ndim != 2:
        raise ValueError(
            f"{file_path}: '{name}' must be 2D after normalization, got shape {array.shape}"
        )
    return array.astype(np.float64, copy=False)


def _coerce_scalar(raw_value, default_value, cast_fn):
    if raw_value is None:
        return default_value
    if isinstance(raw_value, np.ndarray):
        if raw_value.shape == ():
            return cast_fn(raw_value.item())
        flat = raw_value.reshape(-1)
        if len(flat) == 1:
            return cast_fn(flat[0])
    return cast_fn(raw_value)


def _coerce_method_name(raw_name, file_path):
    if raw_name is None:
        return Path(file_path).stem.replace("_predictions", "")
    return _coerce_scalar(raw_name, Path(file_path).stem.replace("_predictions", ""), str)


def _coerce_sample_ids(raw_sample_ids, n_samples, file_path):
    if raw_sample_ids is None:
        return np.array([f"sample_{idx:05d}" for idx in range(n_samples)], dtype=object)
    sample_ids = np.asarray(raw_sample_ids).reshape(-1)
    if len(sample_ids) != n_samples:
        raise ValueError(
            f"{file_path}: 'sample_id' length {len(sample_ids)} does not match n_samples {n_samples}"
        )
    return sample_ids.astype(object)


def load_result_file(file_path, default_rate=SAMPLE_RATE):
    file_path = _as_path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Result file not found: {file_path}")

    with np.load(file_path, allow_pickle=True) as data:
        keys = set(data.files)
        required = {"y_true", "y_pred"}
        missing = required - keys
        if missing:
            missing_str = ", ".join(sorted(missing))
            raise ValueError(f"{file_path}: missing required arrays: {missing_str}")

        y_true = _ensure_2d(data["y_true"], "y_true", file_path)
        y_pred = _ensure_2d(data["y_pred"], "y_pred", file_path)
        if y_true.shape != y_pred.shape:
            raise ValueError(
                f"{file_path}: y_true shape {y_true.shape} does not match y_pred shape {y_pred.shape}"
            )

        method_name = _coerce_method_name(data["method_name"] if "method_name" in keys else None, file_path)
        sample_rate = _coerce_scalar(data["sample_rate"] if "sample_rate" in keys else None, default_rate, int)
        target_mode = _coerce_scalar(data["target_mode"] if "target_mode" in keys else None, TARGET_MODE, str)
        early_ms = _coerce_scalar(data["early_ms"] if "early_ms" in keys else None, EARLY_MS, float)
        output_dim = _coerce_scalar(data["output_dim"] if "output_dim" in keys else None, y_true.shape[1], int)
        sample_ids = _coerce_sample_ids(data["sample_id"] if "sample_id" in keys else None, y_true.shape[0], file_path)

    if target_mode != "early":
        warnings.warn(f"{method_name}: expected target_mode='early', got {target_mode!r}")
    if output_dim != y_true.shape[1]:
        warnings.warn(f"{method_name}: output_dim metadata {output_dim} does not match array length {y_true.shape[1]}")

    return {
        "file_path": file_path,
        "method_name": method_name,
        "sample_rate": sample_rate,
        "target_mode": target_mode,
        "early_ms": early_ms,
        "output_dim": output_dim,
        "sample_id": sample_ids,
        "y_true": y_true,
        "y_pred": y_pred,
    }


def load_all_results(result_files, default_rate=SAMPLE_RATE):
    loaded = []
    for file_path in result_files:
        loaded.append(load_result_file(file_path, default_rate=default_rate))
    return loaded


In [ ]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def r2_score(y_true, y_pred, eps=EPSILON):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot <= eps:
        return np.nan
    return float(1.0 - (ss_res / ss_tot))


def _energy(signal):
    return np.sum(np.square(signal), axis=-1)


def _peak_indices(signal):
    return np.argmax(np.abs(signal), axis=1)


def compute_early_metrics(y_true, y_pred, sample_rate, eps=EPSILON):
    flat_true = y_true.reshape(-1)
    flat_pred = y_pred.reshape(-1)
    per_sample_mae = np.mean(np.abs(y_true - y_pred), axis=1)
    per_sample_rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=1))
    per_sample_r2 = np.array([r2_score(t, p) for t, p in zip(y_true, y_pred)], dtype=np.float64)

    true_energy = _energy(y_true)
    pred_energy = _energy(y_pred)
    energy_abs_error = np.abs(pred_energy - true_energy)
    energy_rel_error = energy_abs_error / np.maximum(true_energy, eps)
    energy_db_error = 10.0 * np.log10(np.maximum(pred_energy, eps) / np.maximum(true_energy, eps))

    true_peak_idx = _peak_indices(y_true)
    pred_peak_idx = _peak_indices(y_pred)
    peak_time_error_ms = np.abs(pred_peak_idx - true_peak_idx) / float(sample_rate) * 1000.0
    true_peak_amp = np.take_along_axis(np.abs(y_true), true_peak_idx[:, None], axis=1).reshape(-1)
    pred_peak_amp = np.take_along_axis(np.abs(y_pred), pred_peak_idx[:, None], axis=1).reshape(-1)
    peak_amp_abs_error = np.abs(pred_peak_amp - true_peak_amp)

    return {
        "early_mae": mae(flat_true, flat_pred),
        "early_rmse": rmse(flat_true, flat_pred),
        "early_r2": r2_score(flat_true, flat_pred),
        "sample_mae_mean": float(np.nanmean(per_sample_mae)),
        "sample_mae_std": float(np.nanstd(per_sample_mae)),
        "sample_rmse_mean": float(np.nanmean(per_sample_rmse)),
        "sample_rmse_std": float(np.nanstd(per_sample_rmse)),
        "sample_r2_mean": float(np.nanmean(per_sample_r2)),
        "sample_r2_std": float(np.nanstd(per_sample_r2)),
        "early_energy_true_mean": float(np.nanmean(true_energy)),
        "early_energy_pred_mean": float(np.nanmean(pred_energy)),
        "early_energy_abs_error_mean": float(np.nanmean(energy_abs_error)),
        "early_energy_relative_error_mean": float(np.nanmean(energy_rel_error)),
        "early_energy_db_error_mean": float(np.nanmean(energy_db_error)),
        "peak_time_error_ms_mean": float(np.nanmean(peak_time_error_ms)),
        "peak_time_error_ms_std": float(np.nanstd(peak_time_error_ms)),
        "peak_amp_abs_error_mean": float(np.nanmean(peak_amp_abs_error)),
        "n_samples": int(y_true.shape[0]),
        "n_points": int(y_true.shape[1]),
        "early_window_ms": float(y_true.shape[1] / float(sample_rate) * 1000.0),
    }


In [ ]:
def cumulative_energy_db(rir, eps=EPSILON):
    rir = np.asarray(rir, dtype=np.float64)
    energy = np.cumsum(np.square(rir))
    total = max(float(energy[-1]) if len(energy) else 0.0, eps)
    return 10.0 * np.log10(np.maximum(energy / total, eps))


def build_sample_metric_rows(method_result, eps=EPSILON):
    rows = []
    sample_rate = method_result["sample_rate"]
    for idx, (sample_id, y_true, y_pred) in enumerate(
        zip(method_result["sample_id"], method_result["y_true"], method_result["y_pred"])
    ):
        true_energy = float(np.sum(np.square(y_true)))
        pred_energy = float(np.sum(np.square(y_pred)))
        true_peak_idx = int(np.argmax(np.abs(y_true)))
        pred_peak_idx = int(np.argmax(np.abs(y_pred)))
        rows.append({
            "method_name": method_result["method_name"],
            "sample_id": sample_id,
            "sample_index": idx,
            "sample_rmse": rmse(y_true, y_pred),
            "sample_mae": mae(y_true, y_pred),
            "sample_r2": r2_score(y_true, y_pred),
            "true_early_energy": true_energy,
            "pred_early_energy": pred_energy,
            "early_energy_abs_error": abs(pred_energy - true_energy),
            "early_energy_relative_error": abs(pred_energy - true_energy) / max(true_energy, eps),
            "early_energy_db_error": 10.0 * np.log10(max(pred_energy, eps) / max(true_energy, eps)),
            "true_peak_time_ms": true_peak_idx / float(sample_rate) * 1000.0,
            "pred_peak_time_ms": pred_peak_idx / float(sample_rate) * 1000.0,
            "peak_time_error_ms": abs(pred_peak_idx - true_peak_idx) / float(sample_rate) * 1000.0,
        })
    return pd.DataFrame(rows)


In [ ]:
loaded_methods = load_all_results(RESULT_FILES, default_rate=SAMPLE_RATE) if RESULT_FILES else []

early_summaries = []
sample_detail_frames = []

for method_result in loaded_methods:
    early_summary = compute_early_metrics(
        method_result["y_true"],
        method_result["y_pred"],
        method_result["sample_rate"],
    )
    early_summary.update({
        "method_name": method_result["method_name"],
        "sample_rate": method_result["sample_rate"],
        "target_mode": method_result["target_mode"],
        "early_ms": method_result["early_ms"],
        "output_dim": method_result["output_dim"],
        "file_path": str(method_result["file_path"]),
    })
    early_summaries.append(early_summary)
    sample_detail_frames.append(build_sample_metric_rows(method_result))

early_summary_df = pd.DataFrame(early_summaries)
sample_details_df = pd.concat(sample_detail_frames, ignore_index=True) if sample_detail_frames else pd.DataFrame()
combined_summary_df = early_summary_df.copy()

print(f"Loaded methods: {len(loaded_methods)}")
if loaded_methods:
    print(f"Evaluation focus: first {EARLY_MS:g} ms early RIR window")
else:
    print("Add early .npz files to RESULT_FILES and rerun the notebook.")


In [ ]:
display_cols_early = [
    "method_name",
    "early_rmse",
    "early_mae",
    "early_r2",
    "early_energy_abs_error_mean",
    "early_energy_relative_error_mean",
    "early_energy_db_error_mean",
    "peak_time_error_ms_mean",
    "peak_amp_abs_error_mean",
    "n_samples",
    "n_points",
    "early_window_ms",
]

expected_method_names = [Path(file_path).stem.replace("_predictions", "") for file_path in RESULT_FILES]
loaded_method_names = [method_result["method_name"] for method_result in loaded_methods]
missing_method_names = [name for name in expected_method_names if name not in loaded_method_names]
unexpected_method_names = [name for name in loaded_method_names if name not in expected_method_names]
if missing_method_names:
    print(f"Missing expected result files or method names: {missing_method_names}")
if unexpected_method_names:
    print(f"Loaded unexpected method names: {unexpected_method_names}")

best_method_name = None
if early_summary_df.empty:
    print("early_summary_df is empty.")
else:
    if BEST_MODEL_METRIC not in early_summary_df.columns:
        raise KeyError(f"BEST_MODEL_METRIC '{BEST_MODEL_METRIC}' not found in early_summary_df")
    early_summary_df = early_summary_df.sort_values(BEST_MODEL_METRIC).reset_index(drop=True)
    combined_summary_df = early_summary_df.copy()
    best_method_name = str(early_summary_df.iloc[0]["method_name"])
    best_metric_value = float(early_summary_df.iloc[0][BEST_MODEL_METRIC])
    print(f"Best method by {BEST_MODEL_METRIC}: {best_method_name} ({best_metric_value:.6f})")
    display(early_summary_df[display_cols_early])

if not sample_details_df.empty:
    display(sample_details_df.head(20))


In [ ]:
if PLOT_EXAMPLES and loaded_methods:
    method_lookup = {method_result["method_name"]: method_result for method_result in loaded_methods}
    methods_to_plot = []
    if best_method_name and best_method_name in method_lookup:
        methods_to_plot.append(method_lookup[best_method_name])
    if PLOT_ALL_MODELS:
        for method_result in loaded_methods:
            if method_result["method_name"] != best_method_name:
                methods_to_plot.append(method_result)

    for method_result in methods_to_plot:
        example_index = min(EXAMPLE_INDEX, method_result["y_true"].shape[0] - 1)
        sample_id = method_result["sample_id"][example_index]
        y_true_example = method_result["y_true"][example_index]
        y_pred_example = method_result["y_pred"][example_index]
        time_axis_ms = np.arange(len(y_true_example)) / method_result["sample_rate"] * 1000.0
        method_summary = early_summary_df.loc[
            early_summary_df["method_name"] == method_result["method_name"]
        ].iloc[0]

        fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
        axes[0].plot(time_axis_ms, y_true_example, label="Ground Truth Early RIR", linewidth=1.0)
        axes[0].plot(time_axis_ms, y_pred_example, label="Prediction / Baseline", linewidth=1.0, alpha=0.8)
        title_prefix = "Best Method" if method_result["method_name"] == best_method_name else "Method"
        axes[0].set_title(
            f"{title_prefix}: {method_result['method_name']} | sample_id={sample_id} | "
            f"early RMSE={method_summary['early_rmse']:.6f}, early R^2={method_summary['early_r2']:.6f}"
        )
        axes[0].set_xlabel("Time (ms)")
        axes[0].set_ylabel("Amplitude")
        axes[0].legend()

        axes[1].plot(time_axis_ms, cumulative_energy_db(y_true_example), label="Ground Truth Cumulative Energy", linewidth=1.2)
        axes[1].plot(time_axis_ms, cumulative_energy_db(y_pred_example), label="Prediction / Baseline Cumulative Energy", linewidth=1.2, alpha=0.8)
        axes[1].set_title(f"Early-Window Cumulative Energy | {method_result['method_name']}")
        axes[1].set_xlabel("Time (ms)")
        axes[1].set_ylabel("Normalized cumulative energy (dB)")
        axes[1].legend()
        plt.show()
else:
    print("Plotting skipped because no methods were loaded or PLOT_EXAMPLES=False.")


In [ ]:
if EXPORT_CSV and not combined_summary_df.empty:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_path = OUTPUT_DIR / "early_measure_summary.csv"
    sample_details_path = OUTPUT_DIR / "early_sample_details.csv"
    combined_summary_df.to_csv(summary_path, index=False)
    if not sample_details_df.empty:
        sample_details_df.to_csv(sample_details_path, index=False)
    print(f"Saved early summary to {summary_path.resolve()}")
    if not sample_details_df.empty:
        print(f"Saved early sample details to {sample_details_path.resolve()}")
else:
    print("CSV export skipped. Set EXPORT_CSV=True after loading early result files.")
